In [ ]:
import pandas as pd
from nilearn.glm.first_level import FirstLevelModel
from nilearn import plotting

from nilearn.datasets import func
from nilearn.plotting import plot_stat_map

from nilearn.glm.first_level import FirstLevelModel
from nilearn.plotting import plot_contrast_matrix, plot_design_matrix

import numpy as np
from nilearn.datasets import fetch_localizer_first_level

from nilearn.plotting import plot_surf_stat_map, show

import nibabel as nib 

from nilearn.image import mean_img

from nilearn.glm.contrasts import compute_fixed_effects

from nilearn.image import mean_img, concat_imgs

from nilearn.glm.contrasts import compute_fixed_effects

from nilearn.reporting import make_glm_report

from sklearn.model_selection import LeaveOneGroupOut
from nilearn.decoding import Decoder
from sklearn.model_selection import KFold

from nilearn.decoding import DecoderRegressor
from nilearn.decoding import FREMClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef
import seaborn as sns
import matplotlib.pyplot as plot_contrast_matrix
import matplotlib.pyplot as plt 

In [ ]:
sub_nbr = 8
directory = f"/home/ec274771/Documents/Tests_preprocessing/S0{sub_nbr}_Preprocessing/Output_fMRIPrep/sub-0{sub_nbr}/ses-01"
t_r = 1.81
mask_img = nib.load(directory + f'/anat/sub-0{sub_nbr}_ses-01_desc-brain_mask.nii.gz')

fmri_imgs = [directory + f"/func/sub-0{sub_nbr}_ses-01_task-VFconf_dir-pa_run-0{run_nb}_space-T1w_desc-preproc_bold.nii.gz" for run_nb in range(1, 9)]

fmri_masks = []
for run_nb in range(1,9):
    fmri_masks = fmri_masks + [directory + f'/func/sub-0{sub_nbr}_ses-01_task-VFconf_dir-pa_run-0{run_nb}_space-T1w_desc-brain_mask.nii.gz']

mean_EPI_img = mean_img(fmri_imgs[0])


mean_EPI_mask = mean_img(fmri_masks[0])

events = {}
labels_confidence = {}
labels_veracity = {}
run_label=[]
for run in range(1, 9):
    run_label.append(run)
    event_run = pd.read_csv(f'/neurospin/metric/EmnaChouria_/MRI_paradigm/External_Participants_test_Paradigm/Responses/Sub0{sub_nbr}_pilote_externe_3T/RUN_{run}/modulation_events.tsv', delimiter='\t', decimal='.')
    event_run['onset'] = event_run['onset'].round(1)
    event_run = event_run[event_run["trial_type"].str.contains("Sentence")]
    events[run] = event_run
    labels_confidence[run] = event_run[event_run["trial_type"] == "response_confidence_Sentence"]["modulation"].values
    labels_veracity[run] = event_run[event_run["trial_type"] == "response_veracity_Sentence"]["modulation"].values

# événements et labels
events_all = []
offset = 0
for run in range(1, 9):
    event_run = events[run].copy()
    event_run['onset'] += offset
    events_all.append(event_run)
    offset += nib.load(fmri_imgs[run - 1]).shape[3] * t_r
events_all = pd.concat(events_all, ignore_index=True)

confidence_labels = np.concatenate([labels_confidence[run] for run in range(1, 9)])
veracity_labels = np.concatenate([labels_veracity[run] for run in range(1, 9)])


events_all_modified = events_all.copy()
confidence_indices = events_all_modified["trial_type"] == "response_confidence_Sentence"
veracity_indices = events_all_modified["trial_type"] == "response_veracity_Sentence"

# identifiant unique pour chaque essai
events_all_modified.loc[confidence_indices, "trial_type"] = [f"confidence_trial_{i}" for i in range(sum(confidence_indices))]
events_all_modified.loc[veracity_indices, "trial_type"] = [f"veracity_trial_{i}" for i in range(sum(veracity_indices))]

confounds_of_interest = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z", "framewise_displacement",
                        "t_comp_cor_00", "t_comp_cor_01", "t_comp_cor_02", #"t_comp_cor_03", # "t_comp_cor_04", "t_comp_cor_05",
                        "a_comp_cor_00", "a_comp_cor_01", "a_comp_cor_02", "a_comp_cor_03"]#, "a_comp_cor_04", "a_comp_cor_05"]
confounds_all = pd.concat([pd.read_csv(directory + f'/func/sub-0{sub_nbr}_ses-01_task-VFconf_dir-pa_run-0{run}_desc-confounds_timeseries.tsv', delimiter='\t')[confounds_of_interest].fillna(0) for run in range(1, 9)], ignore_index=True)

# GLM sur toutes les données
fmri_img_all = concat_imgs(fmri_imgs)
glm = FirstLevelModel(t_r, hrf_model="spm + derivative + dispersion", mask_img=mean_EPI_mask, standardize=True, smoothing_fwhm=5)
glm.fit(fmri_img_all, events=events_all_modified, confounds=confounds_all)

# Générer une z-map pour chaque essai
z_maps_confidence = []
z_maps_veracity = []
for i in range(len(confidence_labels)):
    z_maps_confidence.append(glm.compute_contrast(f"confidence_trial_{i}"))
    z_maps_veracity.append(glm.compute_contrast(f"veracity_trial_{i}"))



In [ ]:
n_runs = 8
samples_per_run = 26
runs_labels = np.repeat(np.arange(1, n_runs + 1), samples_per_run)

test_run = 1
is_test = (runs_labels == test_run)

X_train, X_test = z_maps_veracity[~is_test], z_maps_veracity[is_test]
y_train, y_test = veracity_labels[~is_test], veracity_labels[is_test]

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [ ]:
decoder_conf3 = Decoder(estimator="ridge_classifier",mask=mean_EPI_mask, screening_percentile= 10, standardize=True, cv=LeaveOneGroupOut())
decoder_conf3.fit(z_maps_confidence_train, confidence_labels_train, groups=runs_labels_train)
acc_conf3 = np.mean(list(decoder_conf3.cv_scores_.values()))
print(f"Confidence accuracy: {acc_conf3:.4f} / Chance: {1/3:.4f}")
pred_conf_labels = decoder_conf3.predict(z_maps_confidence_test)
accuracy = accuracy_score(confidence_labels_test, pred_conf_labels)
print(f"Accuracy on the test set: {accuracy: .3f}")

cm = confusion_matrix(confidence_labels_test, pred_conf_labels)


classes = np.unique(np.concatenate((confidence_labels_test, pred_conf_labels)))

class_labels = {1: "Incertain", 2: "Probable", 3: "Certain"}

# Création du heatmap avec les bonnes annotations
sns.heatmap(cm, annot=False, fmt="d", cmap="Blues", 
            xticklabels=[class_labels[c] for c in classes], 
            yticklabels=[class_labels[c] for c in classes])

plt.xlabel("Prédit")
plt.ylabel("Vrai")
plt.title("Matrice de confusion - Ridge classifier")
plt.show()